# GOES ABI True Color with a user-defined domain

## Goal

Load GOES ABI files, enter a geographic domain in decimal longitude/latitude, create a True Color composite, save it as a PNG, and display the image inside JupyterLab.

The example domain around Shishaldin is `(-166.0, 54.0, -162.0, 56.0)`. It is editable input, not an automatic preset.

### Reference image

The image below is a reference GOES-18 Full Disk view. The image generated from your files appears near the end of the notebook.

![Reference GOES-18 imagery](https://rmsm95.github.io/GOES-NESDIS_downlaoder/assets/goes18-header.jpg)

Reference imagery: [NOAA/NASA GOES-18](https://commons.wikimedia.org/wiki/File:GOES-18_full_disk_GeoColor_image_from_May_5,_2022.png).

## Setup

Run the next cell once in a new environment. Restart the kernel after installation if Jupyter asks you to.

In [ ]:
%pip install -r ../requirements-notebooks.txt

In [ ]:
from pathlib import Path
import sys

from IPython.display import Image, display
from PIL import Image as PillowImage
from satpy import Scene

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from examples.render_satellite import crop_and_resample_scene, expand_inputs, validate_bbox

## Steps

### 1. Enter the files, RGB, and domain

Write all four domain values as decimal degrees. The order is `MIN_LON, MIN_LAT, MAX_LON, MAX_LAT`.

In [ ]:
FILES_GLOB = str(REPO_ROOT / "data" / "goes" / "*.nc")
COMPOSITE = "true_color"

# User-defined Shishaldin example in decimal degrees.
DOMAIN = (-166.0, 54.0, -162.0, 56.0)

OUTPUT = REPO_ROOT / "output" / "goes_shishaldin_true_color.png"

validate_bbox(DOMAIN)

### 2. Find and check the GOES files

For True Color, keep `C01`, `C02`, and `C03` from the same scan together. Shishaldin normally requires GOES Full Disk (`F`) source files.

In [ ]:
files = expand_inputs([FILES_GLOB])
if not files:
    raise FileNotFoundError(
        f"No GOES NetCDF files matched {FILES_GLOB}. "
        "Download matching ABI channels first."
    )

print(f"Found {len(files)} GOES files")
for filename in files[:8]:
    print(Path(filename).name)

### 3. Load the RGB composite

In [ ]:
scene = Scene(reader="abi_l1b", filenames=files)
available = {str(name) for name in scene.available_composite_names()}

selected_composite = COMPOSITE
if selected_composite not in available and selected_composite == "true_color":
    if "true_color_raw" in available:
        selected_composite = "true_color_raw"

if selected_composite not in available:
    raise ValueError(
        f"{COMPOSITE!r} is unavailable. Check that all required channels "
        "belong to the same scan."
    )

scene.load([selected_composite], generate=True)
print(f"Loaded composite: {selected_composite}")

### 4. Crop, resample, and save

In [ ]:
output_scene = crop_and_resample_scene(scene, domain=DOMAIN)
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
output_scene.save_dataset(
    selected_composite,
    filename=str(OUTPUT),
    writer="simple_image",
)
print(f"Image created: {OUTPUT.resolve()}")

### 5. Display the generated satellite image

In [ ]:
display(Image(filename=str(OUTPUT)))

## Checks

Confirm that the output exists, has nonzero dimensions, contains the requested region, and uses channels from one observation.

In [ ]:
if not OUTPUT.exists():
    raise FileNotFoundError(OUTPUT)

with PillowImage.open(OUTPUT) as image:
    print(f"PNG size: {image.width} x {image.height} pixels")
    assert image.width > 0 and image.height > 0

## Next steps

- Change all four decimal `DOMAIN` values for another study area.
- Change `COMPOSITE` to `natural_color`, `airmass`, or another composite supported by the downloaded channels.
- Omit cropping only when the complete source extent is required. A real Full Disk output requires ABI `F` input files.